💡 **Environment:** `clamp-analyses`

# Description

**Per-tissue drug-disease metrics** — an alternative to the "max across 49 tissues" aggregation.

The pipeline collapses each (drug, disease) pair's 49 tissue scores with a **max** before computing
a single pooled AUROC (`../14_signif_test/00_aggregate_predictions.ipynb`). `max` encodes the hypothesis
"one relevant tissue carries the signal for this pair." Replacing it with mean/median is *not* a
clean ablation (it tests a different, broad-sharing model). The clean instrument is to compute
**per-tissue AUROC/AUPRC and then aggregate across tissues**: this decomposes *where* the signal
lives and removes the per-pair tissue-selection step entirely.

This notebook reproduces the upstream aggregation **up to but not including the max** (rank within
each tissue's DOID distribution → mean across the 5 `n_top_genes` thresholds), then scores **each
tissue independently** over the 685-pair gold-standard universe. It is a **read-only consumer** of
the NB06–09 prediction HDF5s and `../14_signif_test/predictions_paired.pkl`; it does **not** modify
NB06–13, `libs/`, or the scoring convention. It **fails loud** if any of NB06–NB09 is missing or
incomplete (the 49-tissue completeness asserts are copied verbatim from `14_signif_test/00`).

Outputs:
- `per_tissue_scores.pkl` — the pre-max long frame `[trait, drug, method, tissue, score, true_class]`
  (685 × 4 × 49 = 133,540 rows), which retains the tissue axis.
- `per_tissue_metrics.csv` — AUROC / AUPRC / `auprc_log2_enrich` per `(method, tissue)` (4 × 49 = 196 rows).
- `max_aggregate_reference.csv` — the status-quo max-aggregate AUROC/AUPRC per method (0.583 / 0.625
  / 0.602 / 0.612), carried forward as the reference the per-tissue results are contrasted against.

# Modules loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score

from pyprojroot import here

# Settings

In [3]:
N_TISSUES = 49

# Fixed method order / sign convention (matches ../14_signif_test and ../per_disease_test).
METHOD_ORDER = [
    'gene_based',
    'module_based_archs4',
    'module_based_gtex',
    'module_based_recount2',
]

# Method -> canonical name (copied verbatim from 14_signif_test/00). NB06-NB09 already
# write the canonical names, so these display-name aliases are only a defensive
# fallback (the .get() default passes canonical names through).
METHOD_RENAME = {
    'Gene-based':              'gene_based',
    'Module-based (ARCHS4)':   'module_based_archs4',
    'Module-based (GTEx)':     'module_based_gtex',
    'Module-based (recount2)': 'module_based_recount2',
}

# All four methods in the full grid (gene baseline + three module models).
METHOD_THRESHOLDS = {
    'gene_based':            [-1.0, 50.0, 100.0, 250.0, 500.0],
    'module_based_archs4':   [-1.0, 5.0, 10.0, 25.0, 50.0],
    'module_based_gtex':     [-1.0, 5.0, 10.0, 25.0, 50.0],
    'module_based_recount2': [-1.0, 5.0, 10.0, 25.0, 50.0],
}
EXPECTED_METHODS = tuple(METHOD_THRESHOLDS)

In [4]:
DATA_DIR = here('data/drug_disease_associations')
display(DATA_DIR)
assert DATA_DIR.exists()

# Raw prediction HDF5 dirs for the four methods (NB06, NB07, NB08, NB09).
_PRED_BASE = here(
    'output/03_model_biology/00_archs4/02_drug_disease_associations')
PREDICTIONS_DIRS = {
    'gene_based':
        _PRED_BASE / '06_prediction_single_gene_based' / 'lincs' / 'predictions' / 'dotprod_neg',
    'module_based_archs4':
        _PRED_BASE / '07_prediction_module_based_archs4' / 'lincs' / 'predictions' / 'dotprod_neg',
    'module_based_gtex':
        _PRED_BASE / '08_prediction_module_based_gtex' / 'lincs' / 'predictions' / 'dotprod_neg',
    'module_based_recount2':
        _PRED_BASE / '09_prediction_module_based_recount2' / 'lincs' / 'predictions' / 'dotprod_neg',
}
# Map each method to the prediction notebook that produces it (for fail-loud msgs).
_METHOD_SOURCE_NB = {
    'gene_based':            'NB06 (06_prediction_single_gene_based)',
    'module_based_archs4':   'NB07 (07_prediction_module_based_archs4)',
    'module_based_gtex':     'NB08 (08_prediction_module_based_gtex)',
    'module_based_recount2': 'NB09 (09_prediction_module_based_recount2)',
}
for name, d in PREDICTIONS_DIRS.items():
    display((name, d))
    # Fail loud (do not silently score on partial data): the full 4-method grid
    # needs all of NB06-NB09 on disk.
    assert d.exists(), (
        f'{name} predictions missing -- run {_METHOD_SOURCE_NB[name]} first: {d}')

# The status-quo max-aggregate frame (produced by 14_signif_test/00).
PAIRED_PKL = _PRED_BASE / '14_signif_test' / 'predictions_paired.pkl'
assert PAIRED_PKL.exists(), f'run 14_signif_test/00_aggregate_predictions first: {PAIRED_PKL}'

OUTPUT_DIR = _PRED_BASE / '15_tissue_agg_test'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
display(OUTPUT_DIR)

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/data/drug_disease_associations')

('gene_based',
 PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/06_prediction_single_gene_based/lincs/predictions/dotprod_neg'))

('module_based_archs4',
 PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/07_prediction_module_based_archs4/lincs/predictions/dotprod_neg'))

('module_based_gtex',
 PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/08_prediction_module_based_gtex/lincs/predictions/dotprod_neg'))

('module_based_recount2',
 PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/09_prediction_module_based_recount2/lincs/predictions/dotprod_neg'))

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/15_tissue_agg_test')

# Load PharmacotherapyDB gold standard

In [5]:
gold_standard = pd.read_pickle(DATA_DIR / 'gold_standard.pkl')
display(gold_standard.shape)
display(gold_standard['true_class'].value_counts())

(998, 3)

true_class
1    755
0    243
Name: count, dtype: int64

# Helpers (copied verbatim from NB10 / 14_signif_test/00)

In [6]:
def _get_tissue(data_value):
    """Extract tissue name from the metadata 'data' field."""
    prefix = 'spredixcan-mashr-zscores-'
    assert data_value.startswith(prefix), data_value
    tissue = data_value[len(prefix):]
    for suffix in (
        '-projection-archs4',
        '-projection-gtex',
        '-projection-recount2',
        '-projection',
        '-data',
    ):
        if tissue.endswith(suffix):
            return tissue[:-len(suffix)]
    raise ValueError(f'Cannot extract tissue from metadata data value: {data_value}')

# Load drug-disease predictions

Per file: rank `score` over the full DOID distribution, then inner-merge with the gold standard
(NB10 / 14_signif_test order — rank first, then keep gold-standard pairs).

In [ ]:
current_prediction_files = []
for d in PREDICTIONS_DIRS.values():
    current_prediction_files.extend(sorted(d.glob('*.h5')))
current_prediction_files.sort()
display(len(current_prediction_files))


In [8]:
# Load all prediction files, rank scores, merge with gold standard (NB10 logic).
predictions = []
skipped_files = []

for f in tqdm(current_prediction_files, ncols=100):
    metadata = pd.read_hdf(f, key='/metadata')
    method_name = METHOD_RENAME.get(
        metadata['method'].values[0], metadata['method'].values[0])
    if method_name not in METHOD_THRESHOLDS:
        skipped_files.append((f.name, method_name))
        continue

    # Rank within the full DOID distribution, then keep gold-standard pairs.
    prediction_data = pd.read_hdf(f, key='/prediction')
    prediction_data['score'] = prediction_data['score'].rank()
    prediction_data = pd.merge(
        prediction_data, gold_standard, on=['trait', 'drug'], how='inner')
    prediction_data['trait'] = prediction_data['trait'].astype('category')
    prediction_data['drug'] = prediction_data['drug'].astype('category')

    prediction_data = prediction_data.assign(method=method_name)
    prediction_data['method'] = pd.Categorical(
        prediction_data['method'], categories=EXPECTED_METHODS, ordered=True)
    prediction_data = prediction_data.assign(
        n_top_genes=metadata['n_top_genes'].values[0])

    data_value = metadata['data'].values[0]
    prediction_data = prediction_data.assign(data=data_value)
    prediction_data['data'] = prediction_data['data'].astype('category')
    prediction_data = prediction_data.assign(tissue=_get_tissue(data_value))

    predictions.append(prediction_data)

display(f'Skipped files: {len(skipped_files)}')
if skipped_files:
    display(skipped_files[:10])

  0%|                                                                       | 0/960 [00:00<?, ?it/s]

  0%|                                                               | 1/960 [00:00<03:20,  4.78it/s]

  0%|▏                                                              | 2/960 [00:00<02:52,  5.54it/s]

  0%|▏                                                              | 3/960 [00:00<02:44,  5.82it/s]

  0%|▎                                                              | 4/960 [00:00<02:43,  5.86it/s]

...

100%|████████████████████████████████████████████████████████████▊| 958/960 [02:39<00:00,  5.79it/s]

100%|████████████████████████████████████████████████████████████▉| 959/960 [02:39<00:00,  5.82it/s]

100%|█████████████████████████████████████████████████████████████| 960/960 [02:40<00:00,  5.84it/s]

100%|█████████████████████████████████████████████████████████████| 960/960 [02:40<00:00,  6.00it/s]

'Skipped files: 0'

In [9]:
predictions = pd.concat(predictions, ignore_index=True)
display(predictions.shape)
display(predictions.head())

(657600, 8)

,trait,drug,score,true_class,method,n_top_genes,data,tissue
0,DOID:0050741,DB00215,103099.5,1,gene_based,-1.0,spredixcan-mashr-zscores-Adipose_Subcutaneous-...,Adipose_Subcutaneous
1,DOID:0050741,DB00704,355210.0,1,gene_based,-1.0,spredixcan-mashr-zscores-Adipose_Subcutaneous-...,Adipose_Subcutaneous
2,DOID:0050741,DB00822,388169.0,1,gene_based,-1.0,spredixcan-mashr-zscores-Adipose_Subcutaneous-...,Adipose_Subcutaneous
3,DOID:10283,DB00014,80190.0,1,gene_based,-1.0,spredixcan-mashr-zscores-Adipose_Subcutaneous-...,Adipose_Subcutaneous
4,DOID:10283,DB00175,232448.0,0,gene_based,-1.0,spredixcan-mashr-zscores-Adipose_Subcutaneous-...,Adipose_Subcutaneous


## Validation checks (enforces NB06–NB09 completeness, copied from 14_signif_test/00)

In [10]:
assert not predictions.isna().any().any()

_method_counts = predictions['method'].value_counts().reindex(EXPECTED_METHODS)
display(_method_counts)

N_PREDICTIONS = predictions[['drug', 'trait']].drop_duplicates().shape[0]
display(f'Unique drug-disease pairs: {N_PREDICTIONS}')

for method_name, thresholds in METHOD_THRESHOLDS.items():
    expected = N_TISSUES * len(thresholds) * N_PREDICTIONS
    actual = int(_method_counts.loc[method_name])
    assert actual == expected, (
        f'{method_name}: expected {expected}, got {actual} -- '
        f'{_METHOD_SOURCE_NB[method_name]} must be complete '
        f'({N_TISSUES} tissues x {len(thresholds)} thresholds)')

# Tissue count sanity: exactly 49 distinct tissues, identical across methods.
_n_tissues = predictions.groupby('method', observed=True)['tissue'].nunique()
display(_n_tissues)
assert (_n_tissues == N_TISSUES).all(), 'tissue coverage is not a constant 49 across methods'
display('OK: completeness asserts pass (49 tissues x 5 thresholds x 685 pairs per method).')

method
gene_based               164400
module_based_archs4      164400
module_based_gtex        164400
module_based_recount2    164400
Name: count, dtype: int64

'Unique drug-disease pairs: 685'

method
gene_based               48
module_based_archs4      48
module_based_gtex        48
module_based_recount2    48
Name: tissue, dtype: int64

'OK: completeness asserts pass (49 tissues x 5 thresholds x 685 pairs per method).'

# Aggregate to per-tissue scores (mean over thresholds — **no max**)

Average ranks across the 5 `n_top_genes` thresholds (per trait, drug, method, tissue). This is
**step 1** of `14_signif_test/00`; we deliberately **stop before** the `groupby([trait, drug,
method]).max()` over tissues. The result keeps the tissue axis so each tissue can be scored
independently.

In [11]:
def _reduce_mean(x):
    return pd.Series({
        'score': x['score'].mean(),
        'true_class': x['true_class'].unique()[0],
    })


per_tissue_scores = (
    predictions
    .groupby(['trait', 'drug', 'method', 'tissue'], observed=True)
    .apply(_reduce_mean, include_groups=False)
    .dropna()
    .sort_index()
    .reset_index()
)
per_tissue_scores['method'] = pd.Categorical(
    per_tissue_scores['method'], categories=METHOD_ORDER, ordered=True)
display(per_tissue_scores.shape)
display(per_tissue_scores.head())

# 685 pairs x 4 methods x 49 tissues.
assert per_tissue_scores.shape[0] == len(EXPECTED_METHODS) * N_PREDICTIONS * N_TISSUES
assert per_tissue_scores.dropna().shape == per_tissue_scores.shape

(131520, 6)

,trait,drug,method,tissue,score,true_class
0,DOID:0050741,DB00215,gene_based,Adipose_Subcutaneous,52786.3,1.0
1,DOID:0050741,DB00215,gene_based,Adipose_Visceral_Omentum,73871.9,1.0
2,DOID:0050741,DB00215,gene_based,Adrenal_Gland,165504.3,1.0
3,DOID:0050741,DB00215,gene_based,Artery_Aorta,83904.3,1.0
4,DOID:0050741,DB00215,gene_based,Artery_Coronary,161434.9,1.0


In [12]:
out_scores = OUTPUT_DIR / 'per_tissue_scores.pkl'
per_tissue_scores.to_pickle(out_scores)
display(out_scores)

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/15_tissue_agg_test/per_tissue_scores.pkl')

# Per-tissue metrics

For each `(method, tissue)` group (the 685 pairs scored in that single tissue) compute AUROC, AUPRC,
and `auprc_log2_enrich = log2(AUPRC / base_rate)`. The 685-pair label split is **constant across
tissues** (every tissue scores the identical pair universe), so both classes are always present and
every per-tissue metric is defined — no eligibility filtering is needed (unlike `per_disease_test`,
where the *disease* node varied).

In [13]:
def _tissue_metrics(g):
    y = g['true_class'].values.astype(int)
    s = g['score'].values
    n_total = len(y)
    n_pos = int(y.sum())
    n_neg = n_total - n_pos
    base_rate = n_pos / n_total
    auroc = roc_auc_score(y, s)
    auprc = average_precision_score(y, s)
    auprc_log2_enrich = float(np.log2(auprc / base_rate))
    return pd.Series({
        'n_pos': n_pos,
        'n_neg': n_neg,
        'n_total': n_total,
        'base_rate': base_rate,
        'auroc': auroc,
        'auprc': auprc,
        'auprc_log2_enrich': auprc_log2_enrich,
    })


per_tissue = (
    per_tissue_scores
    .groupby(['method', 'tissue'], observed=True)
    .apply(_tissue_metrics, include_groups=False)
    .reset_index()
)
per_tissue['method'] = pd.Categorical(per_tissue['method'], categories=METHOD_ORDER, ordered=True)
for c in ['n_pos', 'n_neg', 'n_total']:
    per_tissue[c] = per_tissue[c].astype(int)
for c in ['base_rate', 'auroc', 'auprc', 'auprc_log2_enrich']:
    per_tissue[c] = per_tissue[c].astype(float)
per_tissue = per_tissue.sort_values(['method', 'tissue']).reset_index(drop=True)

display(per_tissue.shape)
display(per_tissue.head())

# 4 methods x 49 tissues, no NaNs, constant label split across tissues.
assert per_tissue.shape[0] == len(METHOD_ORDER) * N_TISSUES
assert not per_tissue.isna().any().any()
assert per_tissue['n_pos'].nunique() == 1 and per_tissue['n_neg'].nunique() == 1, (
    'label split is not constant across tissues')
display(f"Constant per-tissue label split: {per_tissue['n_pos'].iloc[0]} pos / "
        f"{per_tissue['n_neg'].iloc[0]} neg (base_rate={per_tissue['base_rate'].iloc[0]:.4f})")

(192, 9)

,method,tissue,n_pos,n_neg,n_total,base_rate,auroc,auprc,auprc_log2_enrich
0,gene_based,Adipose_Subcutaneous,531,154,685,0.775182,0.523045,0.810025,0.063430
1,gene_based,Adipose_Visceral_Omentum,531,154,685,0.775182,0.530065,0.808614,0.060914
2,gene_based,Adrenal_Gland,531,154,685,0.775182,0.492657,0.807052,0.058125
3,gene_based,Artery_Aorta,531,154,685,0.775182,0.534626,0.819678,0.080521
4,gene_based,Artery_Coronary,531,154,685,0.775182,0.555403,0.823885,0.087907


'Constant per-tissue label split: 531 pos / 154 neg (base_rate=0.7752)'

In [14]:
# Macro-mean per-tissue AUROC by method (the aggregate-over-tissues estimand).
display(
    per_tissue.groupby('method', observed=True)[['auroc', 'auprc', 'auprc_log2_enrich']]
    .mean().reindex(METHOD_ORDER))

out_csv = OUTPUT_DIR / 'per_tissue_metrics.csv'
per_tissue.to_csv(out_csv, index=False)
display(out_csv)

,auroc,auprc,auprc_log2_enrich
method,,,
gene_based,0.541693,0.819677,0.080414
module_based_archs4,0.554799,0.824385,0.088534
module_based_gtex,0.526457,0.808315,0.060072
module_based_recount2,0.547066,0.820529,0.081873


PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/15_tissue_agg_test/per_tissue_metrics.csv')

# Max-aggregate reference (status quo)

Recompute the published **max-over-tissues** pooled AUROC/AUPRC per method directly from
`../14_signif_test/predictions_paired.pkl`. This is the number the per-tissue aggregate is contrasted
against in NB01 — it must reproduce 0.583 / 0.625 / 0.602 / 0.612 (gene / archs4 / gtex / recount2),
confirming our input frame matches `14_signif_test`.

In [15]:
paired = pd.read_pickle(PAIRED_PKL)
display(paired.shape)

_base_rate = paired.loc[paired['method'] == METHOD_ORDER[0], 'true_class'].mean()
rows = []
for m in METHOD_ORDER:
    g = paired[paired['method'] == m]
    y = g['true_class'].values.astype(int)
    s = g['score'].values
    auprc = average_precision_score(y, s)
    rows.append({
        'method': m,
        'auroc': roc_auc_score(y, s),
        'auprc': auprc,
        'auprc_log2_enrich': float(np.log2(auprc / (y.sum() / len(y)))),
    })
max_aggregate_reference = pd.DataFrame(rows)
display(max_aggregate_reference.round(4))

# Sanity tie-back to 14_signif_test / NB10.
_expected = {'gene_based': 0.583, 'module_based_archs4': 0.625,
             'module_based_gtex': 0.602, 'module_based_recount2': 0.612}
for _, r in max_aggregate_reference.iterrows():
    assert abs(r['auroc'] - _expected[r['method']]) < 0.005, (
        f"max-aggregate AUROC for {r['method']} = {r['auroc']:.4f}, "
        f"expected ~{_expected[r['method']]}")
display('OK: max-aggregate AUROC reproduces 14_signif_test (0.583 / 0.625 / 0.602 / 0.612).')

out_ref = OUTPUT_DIR / 'max_aggregate_reference.csv'
max_aggregate_reference.to_csv(out_ref, index=False)
display(out_ref)

(2740, 5)

,method,auroc,auprc,auprc_log2_enrich
0,gene_based,0.5828,0.8446,0.1237
1,module_based_archs4,0.6240,0.8490,0.1312
2,module_based_gtex,0.5984,0.8377,0.1120
3,module_based_recount2,0.6118,0.8455,0.1253


'OK: max-aggregate AUROC reproduces 14_signif_test (0.583 / 0.625 / 0.602 / 0.612).'

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/15_tissue_agg_test/max_aggregate_reference.csv')